# Task 1. Dataset and Dependencies

### 1.1. Install and import all required dependencies

In [ ]:
%pip install -q pgserver==0.1.4 psycopg2-binary sqlalchemy pandas

In [ ]:
from pathlib import Path
from google.colab import files

import pandas as pd
import pgserver

from sqlalchemy import create_engine, text, Text

Start and check server

In [ ]:
pg = pgserver.get_server(
    "/tmp/fp_pg",
    cleanup_mode="stop"
)

engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


Add helper function to execute SQL without expected output to show (e.g. create table)

In [ ]:
def run_sql(query):
    with engine.begin() as conn:
        conn.execute(text(query))
        #conn.exec_driver_sql(query)

Add helper function to execute SQL queries and show output (e.g., SELECT query)

In [ ]:
def run_query(query, params=None):
    with engine.connect() as conn:
        return pd.read_sql_query(
            text(query), conn, params=params)

### 1.2. Upload CSV and write data to raw table

Upload CSV file with initial data

In [ ]:
csv_path = Path("/content/infectious_cases.csv")

if not csv_path.exists():
    uploaded = files.upload()

    if not csv_path.exists():
        raise FileNotFoundError(
            "Please upload infectious_cases.csv"
        )

df_raw = pd.read_csv(
    csv_path,
    dtype="string",
    keep_default_na=False
)

print("Dataset shape:", df_raw.shape)
display(df_raw.head())
display(df_raw.dtypes)

Dataset shape: (10521, 12)


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,,6160,0,,,,,,
1,Afghanistan,AFG,1981,,5859,0,,,,,,
2,Afghanistan,AFG,1982,,9730,0,,,,,,
3,Afghanistan,AFG,1983,,13937,0,,,,,,
4,Afghanistan,AFG,1984,,3864,0,,,,,,


,0
Entity,string[python]
Code,string[python]
Year,string[python]
Number_yaws,string[python]
polio_cases,string[python]
cases_guinea_worm,string[python]
Number_rabies,string[python]
Number_malaria,string[python]
Number_hiv,string[python]
Number_tuberculosis,string[python]


Create `pandemic` schema

In [ ]:
with engine.begin() as conn:
    conn.execute(text(
        "CREATE SCHEMA IF NOT EXISTS pandemic"
    ))
    conn.execute(text(
        "SET search_path TO pandemic"
    ))

    current_schema = conn.execute(
        text("SELECT current_schema()")
    ).scalar_one()

    print("Current schema:", current_schema)

Current schema: pandemic


Import data from CSV file into raw table (`pandemic.infectious_cases`). Replace empty values with `NULL`

In [ ]:
# Preserve original values and convert blanks to SQL NULL.
df_import = df_raw.copy()
df_import = df_import.mask(df_import.eq(""), pd.NA)

# Store every original column as TEXT in the raw layer.
raw_types = {
    column: Text()
    for column in df_import.columns
}

df_import.to_sql(
    name="infectious_cases",
    con=engine,
    schema="pandemic",
    if_exists="replace",
    index=False,
    chunksize=2000,
    dtype=raw_types
)

print("Rows imported:", len(df_import))

display(run_query("""
SELECT *
FROM pandemic.infectious_cases
LIMIT 10;
"""))

Rows imported: 10521


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,None,6160,0,None,None,None,None,None,None
1,Afghanistan,AFG,1981,None,5859,0,None,None,None,None,None,None
2,Afghanistan,AFG,1982,None,9730,0,None,None,None,None,None,None
3,Afghanistan,AFG,1983,None,13937,0,None,None,None,None,None,None
4,Afghanistan,AFG,1984,None,3864,0,None,None,None,None,None,None
5,Afghanistan,AFG,1985,None,13867,0,None,None,None,None,None,None
6,Afghanistan,AFG,1986,None,12901,0,None,None,None,None,None,None
7,Afghanistan,AFG,1987,None,4396,0,None,None,None,None,None,None
8,Afghanistan,AFG,1988,None,2149,0,None,None,None,None,None,None
9,Afghanistan,AFG,1989,None,385,0,None,None,None,None,None,None


Check number of rows and some values in the raw table

In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT "Entity") AS unique_entities,
    MIN("Year"::INTEGER) AS min_year,
    MAX("Year"::INTEGER) AS max_year,
    COUNT(*) FILTER (
        WHERE "Code" IS NULL
    ) AS missing_codes,
    COUNT(*) FILTER (
        WHERE "Number_rabies" IS NOT NULL
    ) AS rabies_values
FROM pandemic.infectious_cases;
"""

display(run_query(query))

,total_rows,unique_entities,min_year,max_year,missing_codes,rabies_values
0,10521,245,1920,2023,1128,6780


Check column types

In [ ]:
query = """
SELECT
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'pandemic'
  AND table_name = 'infectious_cases'
ORDER BY ordinal_position;
"""

display(run_query(query))

,column_name,data_type,is_nullable
0,Entity,text,YES
1,Code,text,YES
2,Year,text,YES
3,Number_yaws,text,YES
4,polio_cases,text,YES
5,cases_guinea_worm,text,YES
6,Number_rabies,text,YES
7,Number_malaria,text,YES
8,Number_hiv,text,YES
9,Number_tuberculosis,text,YES


# Task 2. Data normalization

Prepare normalized tables

In [ ]:
query = """
-- Drop dependent table first
DROP TABLE IF EXISTS pandemic.stat;
DROP TABLE IF EXISTS pandemic.entities;

-- Entity dimension
CREATE TABLE pandemic.entities (
    entity_id INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    entity_name TEXT NOT NULL,
    entity_code TEXT,

    CONSTRAINT uq_entity
        UNIQUE NULLS NOT DISTINCT (entity_name, entity_code)
);

-- Disease statistics
CREATE TABLE pandemic.stat (
    stat_id INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    entity_id INTEGER NOT NULL,
    year SMALLINT NOT NULL,

    yaws_number NUMERIC,
    polio_number NUMERIC,
    guinea_worm_number NUMERIC,
    rabies_number NUMERIC,
    malaria_number NUMERIC,
    hiv_number NUMERIC,
    tuberculosis_number NUMERIC,
    smallpox_number NUMERIC,
    cholera_number NUMERIC,

    CONSTRAINT fk_entity
        FOREIGN KEY (entity_id)
        REFERENCES pandemic.entities (entity_id),

    CONSTRAINT uq_entity_year
        UNIQUE (entity_id, year)
);
"""

run_sql(query)

Insert entity records into corresponding table (`entity_id` will be generated)

In [ ]:
query = """
INSERT INTO pandemic.entities (
    entity_name,
    entity_code
)
SELECT DISTINCT
    TRIM("Entity") AS entity_name,
    TRIM("Code") AS entity_code
FROM pandemic.infectious_cases
"""

run_sql(query)

Check correctness of entities insertion

In [ ]:
query = """
SELECT *
FROM pandemic.entities
ORDER BY entity_id
LIMIT 10;
"""

run_query(query)

,entity_id,entity_name,entity_code
0,1,Morocco,MAR
1,2,Iran,IRN
2,3,Russia,RUS
3,4,Democratic Republic of Congo,COD
4,5,Ethiopia,ETH
5,6,Trinidad and Tobago,TTO
6,7,Israel,ISR
7,8,Antigua and Barbuda,ATG
8,9,Libya,LBY
9,10,Guatemala,GTM


Insert stat records, using generated entities' ID and raw data

In [ ]:
query = """
INSERT INTO pandemic.stat (
    entity_id,
    year,
    yaws_number,
    polio_number,
    guinea_worm_number,
    rabies_number,
    malaria_number,
    hiv_number,
    tuberculosis_number,
    smallpox_number,
    cholera_number
)
SELECT
    e.entity_id,
    r."Year"::SMALLINT AS year,
    r."Number_yaws"::NUMERIC AS yaws_number,
    r."polio_cases"::NUMERIC AS polio_number,
    r."cases_guinea_worm"::NUMERIC AS guinea_worm_number,
    r."Number_rabies"::NUMERIC AS rabies_number,
    r."Number_malaria"::NUMERIC AS malaria_number,
    r."Number_hiv"::NUMERIC AS hiv_number,
    r."Number_tuberculosis"::NUMERIC AS tuberculosis_number,
    r."Number_smallpox"::NUMERIC AS smallpox_number,
    r."Number_cholera_cases"::NUMERIC AS cholera_number
FROM pandemic.infectious_cases AS r
INNER JOIN pandemic.entities AS e
    ON TRIM(r."Entity") = e.entity_name
   AND TRIM(r."Code") IS NOT DISTINCT FROM e.entity_code;
"""

run_sql(query)

Check correctness of data insertion

In [ ]:
query = """
SELECT
    (SELECT COUNT(*) FROM pandemic.infectious_cases) AS raw_rows,
    (SELECT COUNT(*) FROM pandemic.entities) AS entities,
    (SELECT COUNT(*) FROM pandemic.stat) AS stats;
"""

run_query(query)

,raw_rows,entities,stats
0,10521,245,10521


In [ ]:
query = """
SELECT COUNT(*) AS unmatched_rows
FROM pandemic.infectious_cases AS r
LEFT JOIN pandemic.entities AS e
    ON r."Entity" = e.entity_name
   AND r."Code" IS NOT DISTINCT FROM e.entity_code
WHERE e.entity_id IS NULL;
"""

run_query(query)

,unmatched_rows
0,0


In [ ]:
query = """
SELECT
    (SELECT COUNT("Number_rabies")
     FROM pandemic.infectious_cases) AS raw_rabies_values,

    (SELECT COUNT(rabies_number)
     FROM pandemic.stat) AS normalized_rabies_values;
"""

run_query(query)

,raw_rabies_values,normalized_rabies_values
0,6780,6780


In [ ]:
query = """
SELECT
    COUNT(*) AS checked_rows,
    COUNT(*) FILTER (
        WHERE
            ROW(
                r."Number_yaws"::NUMERIC,
                r."polio_cases"::NUMERIC,
                r."cases_guinea_worm"::NUMERIC,
                r."Number_rabies"::NUMERIC,
                r."Number_malaria"::NUMERIC,
                r."Number_hiv"::NUMERIC,
                r."Number_tuberculosis"::NUMERIC,
                r."Number_smallpox"::NUMERIC,
                r."Number_cholera_cases"::NUMERIC
            )
            IS DISTINCT FROM
            ROW(
                s.yaws_number,
                s.polio_number,
                s.guinea_worm_number,
                s.rabies_number,
                s.malaria_number,
                s.hiv_number,
                s.tuberculosis_number,
                s.smallpox_number,
                s.cholera_number
            )
    ) AS mismatched_rows
FROM pandemic.infectious_cases AS r
JOIN pandemic.entities AS e
    ON r."Entity" = e.entity_name
   AND r."Code" IS NOT DISTINCT FROM e.entity_code
JOIN pandemic.stat AS s
    ON s.entity_id = e.entity_id
   AND s.year = r."Year"::SMALLINT;
"""

run_query(query)

,checked_rows,mismatched_rows
0,10521,0


In [ ]:
query = """
SELECT *
FROM pandemic.stat
ORDER BY stat_id
LIMIT 10;
"""

run_query(query)

,stat_id,entity_id,year,yaws_number,polio_number,guinea_worm_number,rabies_number,malaria_number,hiv_number,tuberculosis_number,smallpox_number,cholera_number
0,1,143,2019,None,32.0,0.0,3.387213,998712.10,849.66254,23717.326,None,2.0
1,2,143,2018,None,23.0,0.0,3.394850,629438.06,810.92230,23930.367,None,NaN
2,3,143,2017,None,16.0,0.0,3.379239,427513.62,765.24567,23483.064,None,33.0
3,4,143,2016,None,14.0,0.0,3.337423,345016.80,718.01410,23112.873,None,677.0
4,5,143,2015,None,22.0,0.0,3.310823,298038.50,669.63860,23044.637,None,58064.0
5,6,143,2014,None,31.0,0.0,3.330081,294260.03,592.78750,23040.713,None,45481.0
6,7,143,1982,None,9730.0,0.0,NaN,NaN,NaN,NaN,None,NaN
7,8,143,1983,None,13937.0,0.0,NaN,NaN,NaN,NaN,None,NaN
8,9,143,1984,None,3864.0,0.0,NaN,NaN,NaN,NaN,None,NaN
9,10,143,1985,None,13867.0,0.0,NaN,NaN,NaN,NaN,None,NaN


The source dataset was normalized into two related tables: pandemic.entities and pandemic.stat.

The entities table contains 245 unique Entity/Code combinations, while the statistics table contains 10,521 annual disease records. Primary and foreign keys establish the relationship between the tables, and UNIQUE constraints prevent duplicate entities and entity-year records.

All source records were successfully transferred, including records with missing entity codes. The validation queries found no unmatched entities, and the number of non-null rabies values remained unchanged at 6,780.

# Task 3. Data analysis

Calculate average, minimal, maximum, and total number of rabies cases by entity

In [ ]:
query = """
SELECT
    s.entity_id,
    e.entity_name,
    AVG(s.rabies_number) AS average_rabies_cases,
    MIN(s.rabies_number) AS minimal_rabies_cases,
    MAX(s.rabies_number) AS maximum_rabies_cases,
    SUM(s.rabies_number) AS total_rabies_cases
FROM pandemic.stat AS s
JOIN pandemic.entities AS e ON s.entity_id = e.entity_id
WHERE s.rabies_number IS NOT NULL
GROUP BY s.entity_id, e.entity_name
ORDER BY average_rabies_cases DESC
LIMIT 10;
"""

run_query(query)

,entity_id,entity_name,average_rabies_cases,minimal_rabies_cases,maximum_rabies_cases,total_rabies_cases
0,102,World,20192.370367,14075.5080,24744.6580,605771.1110
1,196,Lower Middle Income (WB),15193.959383,10202.5280,19182.7950,455818.7815
2,162,South Asia (WB),11729.888753,7271.2754,15361.8780,351896.6626
3,66,South-East Asia Region (WHO),11424.326893,6806.0127,15641.9580,342729.8068
4,51,G20,10189.046003,6339.0796,13164.8810,305671.3801
5,182,India,8599.173280,5425.8726,11121.1420,257975.1984
6,235,Sub-Saharan Africa (WB),4748.268090,4356.8184,5017.9770,142448.0427
7,82,African Region (WHO),4649.702377,4283.0730,4917.7840,139491.0713
8,65,Low Income (WB),3568.105450,3014.6170,3873.5330,107043.1635
9,11,East Asia & Pacific (WB),3427.461133,2009.9395,4591.0107,102823.8340


The query calculates the average, minimum, maximum, and total reported rabies indicator for each entity with available data. Results are sorted by average value in descending order, with the top 10 entities displayed.

The highest average belongs to World (20,192.37), followed by Lower Middle Income (WB) and South Asia (WB). The ranking includes aggregate regions and economic groups, not only individual countries. Missing values are excluded from the calculations.

# Task 4. Difference in Years

For each annual record, construct January 1 of the observation year, retrieve the current date, and calculate the number of complete years between these dates using PostgreSQL date functions.

In [ ]:
query = """
SELECT
    s.stat_id,
    e.entity_name,
    s.year,
    MAKE_DATE(s.year::INTEGER, 1, 1) AS year_start_date,
    CURRENT_DATE AS today_date,
    EXTRACT(
        YEAR FROM AGE(
            CURRENT_DATE,
            MAKE_DATE(s.year::INTEGER, 1, 1)
        )
    )::INTEGER AS years_elapsed
FROM pandemic.stat AS s
JOIN pandemic.entities AS e
    ON s.entity_id = e.entity_id
ORDER BY s.year DESC, e.entity_name
LIMIT 20;
"""

display(run_query(query))

,stat_id,entity_name,year,year_start_date,today_date,years_elapsed
0,10377,World,2023,2023-01-01,2026-11-01,3
1,39,Afghanistan,2022,2022-01-01,2026-11-01,4
2,170,Albania,2022,2022-01-01,2026-11-01,4
3,214,Algeria,2022,2022-01-01,2026-11-01,4
4,336,Andorra,2022,2022-01-01,2026-11-01,4
5,380,Angola,2022,2022-01-01,2026-11-01,4
6,428,Antigua and Barbuda,2022,2022-01-01,2026-11-01,4
7,442,Argentina,2022,2022-01-01,2026-11-01,4
8,514,Armenia,2022,2022-01-01,2026-11-01,4
9,617,Australia,2022,2022-01-01,2026-11-01,4


Check year number calculation

In [ ]:
query = """
SELECT DISTINCT
    year,
    MAKE_DATE(year::INTEGER, 1, 1) AS year_start_date,
    CURRENT_DATE AS today_date,
    EXTRACT(
        YEAR FROM AGE(
            CURRENT_DATE,
            MAKE_DATE(year::INTEGER, 1, 1)
        )
    )::INTEGER AS years_elapsed
FROM pandemic.stat
WHERE year IN (1920, 1996, 2023)
ORDER BY year;
"""

display(run_query(query))

,year,year_start_date,today_date,years_elapsed
0,1920,1920-01-01,2026-11-01,106
1,1996,1996-01-01,2026-11-01,30
2,2023,2023-01-01,2026-11-01,3


The query uses MAKE_DATE() to construct January 1 of each observation year and CURRENT_DATE to obtain the current date. The combination of AGE() and EXTRACT(YEAR FROM ...) calculates the number of complete elapsed years.

The calculation is performed on the normalized statistics table and does not require modifying stored data. Results are updated automatically based on the date when the query is executed.

# Task 5. User-Defined SQL Functions

### 5.1. Calculate Difference in Years

Create a SQL function that accepts an observation year and returns the number of complete years between January 1 of that year and the current date.

In [ ]:
query = """
CREATE OR REPLACE FUNCTION pandemic.years_since(
    p_year INTEGER
)
RETURNS INTEGER
LANGUAGE SQL
STABLE
RETURNS NULL ON NULL INPUT
AS $$
    SELECT EXTRACT(
        YEAR FROM AGE(
            CURRENT_DATE,
            MAKE_DATE(p_year, 1, 1)
        )
    )::INTEGER;
$$;
"""

run_sql(query)

Check function

In [ ]:
query = """
SELECT
    year,
    MAKE_DATE(year, 1, 1) AS year_start_date,
    CURRENT_DATE AS today_date,
    pandemic.years_since(year) AS years_elapsed
FROM (VALUES (1920), (1996), (2023)) AS test(year)
ORDER BY year;
"""

display(run_query(query))

,year,year_start_date,today_date,years_elapsed
0,1920,1920-01-01,2026-11-01,106
1,1996,1996-01-01,2026-11-01,30
2,2023,2023-01-01,2026-11-01,3


In [ ]:
query = """
SELECT
    COUNT(*) AS checked_rows,
    COUNT(*) FILTER (
        WHERE pandemic.years_since(year::INTEGER)
        IS DISTINCT FROM
        EXTRACT(
            YEAR FROM AGE(
                CURRENT_DATE,
                MAKE_DATE(year::INTEGER, 1, 1)
            )
        )::INTEGER
    ) AS mismatched_rows
FROM pandemic.stat;
"""

display(run_query(query))

,checked_rows,mismatched_rows
0,10521,0


### 5.2. Function 2: Average Cases per Period

Create a SQL function that divides the annual disease indicator by a specified number of periods. Test monthly (12), quarterly (4), and semiannual (2) averages. Handle invalid divisors to avoid division-by-zero errors.

In [ ]:
query = """
CREATE OR REPLACE FUNCTION pandemic.average_cases_per_period(
    p_annual_cases NUMERIC,
    p_divisor INTEGER
)
RETURNS NUMERIC
LANGUAGE SQL
IMMUTABLE
RETURNS NULL ON NULL INPUT
AS $$
    SELECT CASE
        WHEN p_divisor > 0
            THEN p_annual_cases / p_divisor
        ELSE NULL
    END;
$$;
"""

run_sql(query)

Check function

In [ ]:
query = """
SELECT
    pandemic.average_cases_per_period(120, 12) AS monthly,
    pandemic.average_cases_per_period(120, 4) AS quarterly,
    pandemic.average_cases_per_period(120, 2) AS semiannual,
    pandemic.average_cases_per_period(120, 0) AS invalid_divisor;
"""

display(run_query(query))

,monthly,quarterly,semiannual,invalid_divisor
0,10.0,30.0,60.0,None


### 5.3. Use Both Functions on the Dataset

In [ ]:
query = """
SELECT
    e.entity_name,
    s.year,
    pandemic.years_since(s.year::INTEGER) AS years_elapsed,
    s.rabies_number AS annual_rabies_cases,

    ROUND(
        pandemic.average_cases_per_period(s.rabies_number, 12),
        4
    ) AS monthly_average,

    ROUND(
        pandemic.average_cases_per_period(s.rabies_number, 4),
        4
    ) AS quarterly_average,

    ROUND(
        pandemic.average_cases_per_period(s.rabies_number, 2),
        4
    ) AS semiannual_average

FROM pandemic.stat AS s
JOIN pandemic.entities AS e
    ON s.entity_id = e.entity_id

WHERE s.rabies_number IS NOT NULL

ORDER BY s.year DESC, e.entity_name
LIMIT 20;
"""

display(run_query(query))

,entity_name,year,years_elapsed,annual_rabies_cases,monthly_average,quarterly_average,semiannual_average
0,Afghanistan,2019,7,3.387213,0.2823,0.8468,1.6936
1,African Region (WHO),2019,7,4378.748000,364.8957,1094.6870,2189.3740
2,Albania,2019,7,0.014225,0.0012,0.0036,0.0071
3,Algeria,2019,7,1.346161,0.1122,0.3365,0.6731
4,American Samoa,2019,7,0.007448,0.0006,0.0019,0.0037
5,Andorra,2019,7,0.000138,0.0000,0.0000,0.0001
6,Angola,2019,7,7.964379,0.6637,1.9911,3.9822
7,Antigua and Barbuda,2019,7,0.004845,0.0004,0.0012,0.0024
8,Argentina,2019,7,0.567866,0.0473,0.1420,0.2839
9,Armenia,2019,7,0.369019,0.0308,0.0923,0.1845


Two user-defined SQL functions were created and tested successfully.

The years_since() function calculates the number of complete years between January 1 of the input year and the current date. Its results matched the equivalent calculation using built-in PostgreSQL functions for all 10,521 records.

The average_cases_per_period() function calculates monthly, quarterly, and semiannual averages by dividing an annual disease indicator by 12, 4, or 2. Invalid divisors are handled safely.

Both functions were applied to the normalized dataset. Only records with non-null numeric rabies values were used for period averages. These averages are calculated estimates, not separately observed monthly or quarterly statistics.